# MuFu Training

Train MuFu on the five shared seeds (0-4), then look at the losses. Recorded results
and the comparison with RF are in `benchmark-results.ipynb`.


In [ ]:
import sys
from dataclasses import replace
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import pandas as pd
import torch

AI_DIR = Path.cwd()
if AI_DIR.name == "notebooks":
    AI_DIR = AI_DIR.parent
elif AI_DIR.name != "ai":
    AI_DIR = AI_DIR / "ai"
sys.path.insert(0, str(AI_DIR / "src"))

from baseline.benchmark import SEEDS, load_benchmark_split, run_benchmark
from encoders.smiles_encoder import (
    DEFAULT_MODEL_NAME,
    DEFAULT_REVISION,
    SmilesEncoder,
)
from mufu.data.dataset import split_datasets
from mufu.experiment import (
    DEFAULT_CONFIG,
    ENCODER,
    benchmark_params,
    make_fit_predict,
)

## Settings


In [ ]:
# True trains and overwrites this model's rows in results.csv
RUN_TRAINING = False

model_name = "mufu"
seeds = SEEDS  # 0, 1, 2, 3, 4

# override fields here, e.g. lr=5e-4, epochs=30
config = replace(DEFAULT_CONFIG, device="cuda" if torch.cuda.is_available() else "cpu")
print(config)

# params from the encoder's settings, so viewing results doesn't load MolFormer
smiles_settings = SimpleNamespace(
    model_name=DEFAULT_MODEL_NAME, revision=DEFAULT_REVISION, freeze=True
)
params = benchmark_params(config, smiles_settings)

## Train

Only runs when `RUN_TRAINING` is True. Builds the data once, then each seed trains a fresh
MuFu and saves its best checkpoint and loss log. Results go to `results.csv` after each seed.


In [ ]:
if RUN_TRAINING:
    df = load_benchmark_split()
    datasets = split_datasets(df)
    for split in datasets:
        print(f"{split}: {len(datasets[split])} molecules")

    # one frozen smiles encoder for each seed so embedding cache only filled once
    smiles_encoder = SmilesEncoder(freeze=True, max_cache_size=len(df))
    assert benchmark_params(config, smiles_encoder) == params
    fit_predict = make_fit_predict(
        datasets,
        smiles_encoder,
        config,
        model_name=model_name,
        checkpoint_dir=AI_DIR / "checkpoints",
        history_dir=AI_DIR / "outputs" / model_name,
        verbose=True,
    )
    summary = run_benchmark(
        fit_predict,
        model=model_name,
        encoder=ENCODER,
        params=params,
        seeds=seeds,
        df=df,
    )
    columns = ["dataset", "endpoint", "split", "metric", "value", "std", "n_seeds"]
    display(summary[columns].round(4))
else:
    print("RUN_TRAINING is False; skipping training.")

## Loss Curves

Read from the saved loss logs, no retraining. Both train and validation loss are computed
without `pos_weight` so they share the same scale (note that train loss is measured with
dropout and modality dropout on, so it won't match validation loss exactly).


In [ ]:
task_loss_cols = [
    ("val_pic50_loss", "Val pIC50 MSE"),
    ("val_bio_cls_loss", "Val bio_cls BCE"),
    ("val_tox_loss", "Val tox BCE"),
]
for seed in seeds:
    loss_path = AI_DIR / "outputs" / model_name / f"seed_{seed}_loss.csv"
    if not loss_path.exists():
        print(f"No loss log for seed {seed} yet.")
        continue
    history = pd.read_csv(loss_path)
    has_tasks = all(col in history.columns for col, _ in task_loss_cols)
    fig, axes = plt.subplots(
        1, 2 if has_tasks else 1, figsize=(13 if has_tasks else 8, 4), squeeze=False
    )
    ax0 = axes[0, 0]
    ax0.plot(
        history["epoch"], history["train_loss"], label="Training loss (unweighted)"
    )
    ax0.plot(
        history["epoch"], history["val_loss"], label="Validation loss (unweighted)"
    )
    ax0.set_xlabel("Epoch")
    ax0.set_ylabel("Loss")
    ax0.set_title(f"{model_name} seed {seed}: total loss")
    ax0.legend()
    ax0.grid(True, alpha=0.3)
    if has_tasks:
        ax1 = axes[0, 1]
        for col, label in task_loss_cols:
            ax1.plot(history["epoch"], history[col], label=label)
        ax1.set_xlabel("Epoch")
        ax1.set_ylabel("Loss")
        ax1.set_title(f"{model_name} seed {seed}: per-task val loss")
        ax1.legend()
        ax1.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()